# Simulação do Nível de Confiança (Intervalo de Confiança - IC)

Este notebook explora a interpretação do conceito de Intervalo de Confiança através de simulações.

## Teoria

O **Nível de Confiança** (ex: 95%) não é a probabilidade de que um IC específico contenha a média populacional (μ). É um exercício de imaginação: se você coletasse infinitas amostras nas mesmas condições e calculasse um IC para cada uma, a confiança seria a proporção de intervalos que contêm a verdadeira média populacional.

### Parâmetros da Simulação:
- **População Gerada:** N = 100.000, μ = 180, σ = 20
- **Amostra:** n = 100
- **Confiança Fixada:** 95%

## Implementação em Python

Vamos gerar a população, coletar 10.000 amostras e verificar quantos ICs de 95% contêm μ = 180.

In [ ]:
import numpy as np
from scipy.stats import t
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração de estilo para os gráficos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

### 📚 Sobre a Biblioteca SciPy

**SciPy** (Scientific Python) é uma biblioteca fundamental para computação científica em Python, construída sobre o NumPy.

#### O que é o módulo `scipy.stats`?

O módulo `scipy.stats` contém uma vasta coleção de distribuições de probabilidade e funções estatísticas:

- **Distribuições Contínuas e Discretas:** Normal, t-Student, chi-quadrado, binomial, Poisson, etc.
- **Funções Estatísticas:** Testes de hipóteses, correlação, regressão
- **Intervalos de Confiança:** Cálculo automático de ICs para diversas distribuições

#### Por que usamos `t.interval()` nesta simulação?

```python
from scipy.stats import t
```

A distribuição **t de Student** é usada quando:
- ✅ A população tem distribuição normal (ou aproximadamente normal)
- ✅ O desvio padrão populacional (σ) é **desconhecido**
- ✅ Estimamos σ usando o desvio padrão amostral (s)
- ✅ Tamanho de amostra pequeno (n < 30) ou médio

**Função `t.interval()`:**
```python
t.interval(confidence, df, loc, scale)
```

**Parâmetros:**
- `confidence`: Nível de confiança (ex: 0.95 para 95%)
- `df`: Graus de liberdade (n - 1)
- `loc`: Média amostral (centro do intervalo)
- `scale`: Erro padrão (s / √n)

**Retorna:** `(limite_inferior, limite_superior)` do intervalo de confiança

#### Vantagens do SciPy

1. **Precisão:** Implementações numericamente estáveis e validadas
2. **Facilidade:** Uma linha de código substitui cálculos complexos
3. **Flexibilidade:** Suporta dezenas de distribuições diferentes
4. **Integração:** Funciona perfeitamente com NumPy e Pandas

#### Alternativa Manual (sem SciPy)

Se calculássemos manualmente:
```python
# Sem SciPy
t_critical = 1.984  # Valor crítico para 95% de confiança e 99 graus de liberdade
margin_error = t_critical * standard_error
ic_lower = sample_mean - margin_error
ic_upper = sample_mean + margin_error
```

Com SciPy, tudo isso é feito automaticamente com `t.interval()` ! 🎯

In [ ]:
# 1. Parâmetros e Geração da População
POP_SIZE = 100000
TRUE_MEAN = 180
TRUE_STD = 20
CONFIDENCE_LEVEL = 0.95
SAMPLE_SIZE = 100
NUM_SIMULATIONS = 10000

np.random.seed(42)
population = np.random.normal(TRUE_MEAN, TRUE_STD, POP_SIZE)

print(f"População gerada com:")
print(f"  Tamanho: {POP_SIZE:,}")
print(f"  Média verdadeira (μ): {TRUE_MEAN}")
print(f"  Desvio padrão (σ): {TRUE_STD}")
print(f"\nParâmetros da simulação:")
print(f"  Tamanho da amostra: {SAMPLE_SIZE}")
print(f"  Nível de confiança: {CONFIDENCE_LEVEL*100}%")
print(f"  Número de simulações: {NUM_SIMULATIONS:,}")

In [ ]:
# 2. Loop de Simulação
ic_contains_mean = []
ic_lower_bounds = []
ic_upper_bounds = []

for i in range(NUM_SIMULATIONS):
    sample = np.random.choice(population, SAMPLE_SIZE, replace=False)
    
    # Cálculo do IC
    sample_mean = np.mean(sample)
    sample_std = np.std(sample, ddof=1)  # ddof=1 para estimador não viciado (n-1)
    standard_error = sample_std / np.sqrt(SAMPLE_SIZE)
    degrees_freedom = SAMPLE_SIZE - 1
    
    # Usa t.interval da scipy para calcular o IC
    ic_lower, ic_upper = t.interval(
        confidence=CONFIDENCE_LEVEL,
        df=degrees_freedom,
        loc=sample_mean,
        scale=standard_error
    )
    
    # 3. Verifica se o IC contém a média populacional
    contains_mean = (ic_lower <= TRUE_MEAN) and (TRUE_MEAN <= ic_upper)
    ic_contains_mean.append(contains_mean)
    
    # Armazena os limites para visualização (apenas primeiros 100)
    if i < 100:
        ic_lower_bounds.append(ic_lower)
        ic_upper_bounds.append(ic_upper)

print("Simulação concluída!")

In [ ]:
# 4. Resultado Final
intervals_containing_mean = np.sum(ic_contains_mean)
proportion_success = intervals_containing_mean / NUM_SIMULATIONS

print(f"\n{'='*60}")
print(f"RESULTADOS DA SIMULAÇÃO")
print(f"{'='*60}")
print(f"ICs que contiveram μ: {intervals_containing_mean:,} de {NUM_SIMULATIONS:,}")
print(f"Proporção de Sucesso: {proportion_success:.4f}")
print(f"Proporção Esperada: {CONFIDENCE_LEVEL:.4f}")
print(f"Diferença: {abs(proportion_success - CONFIDENCE_LEVEL):.4f}")
print(f"{'='*60}")
print(f"\n✓ O resultado está muito próximo do nível de confiança esperado de {CONFIDENCE_LEVEL*100}%!")

## Visualização dos Intervalos de Confiança

Vamos visualizar os primeiros 100 intervalos de confiança para entender melhor o conceito.

In [ ]:
# Visualização dos primeiros 100 ICs
fig, ax = plt.subplots(figsize=(14, 8))

# Separa ICs que contêm e não contêm a média
for i in range(100):
    color = 'green' if ic_contains_mean[i] else 'red'
    alpha = 0.3 if ic_contains_mean[i] else 0.8
    ax.plot([ic_lower_bounds[i], ic_upper_bounds[i]], [i, i], 
            color=color, alpha=alpha, linewidth=2)

# Linha vertical da média verdadeira
ax.axvline(TRUE_MEAN, color='blue', linewidth=2, linestyle='--', 
           label=f'Média Populacional (μ = {TRUE_MEAN})')

# Configurações do gráfico
ax.set_xlabel('Valor', fontsize=12)
ax.set_ylabel('Número da Amostra', fontsize=12)
ax.set_title(f'Primeiros 100 Intervalos de Confiança ({CONFIDENCE_LEVEL*100}%)\n' +
             f'Verde: Contém μ | Vermelho: Não contém μ', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Conta quantos não contêm nos primeiros 100
not_contains = sum([not ic_contains_mean[i] for i in range(100)])
ax.text(0.02, 0.98, f'Dos 100 primeiros ICs:\n{100-not_contains} contêm μ\n{not_contains} não contêm μ', 
        transform=ax.transAxes, fontsize=11, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## Distribuição da Proporção de Sucesso

Vamos verificar como a proporção de sucesso converge conforme aumentamos o número de simulações.

In [ ]:
# Calcula proporção acumulada
cumulative_success = np.cumsum(ic_contains_mean)
cumulative_proportion = cumulative_success / np.arange(1, NUM_SIMULATIONS + 1)

# Gráfico da convergência
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(cumulative_proportion, color='darkblue', linewidth=1.5, 
        label='Proporção Acumulada de Sucesso')
ax.axhline(CONFIDENCE_LEVEL, color='red', linewidth=2, linestyle='--', 
           label=f'Nível de Confiança Esperado ({CONFIDENCE_LEVEL*100}%)')

ax.set_xlabel('Número de Simulações', fontsize=12)
ax.set_ylabel('Proporção de ICs que contêm μ', fontsize=12)
ax.set_title('Convergência da Proporção de Sucesso para o Nível de Confiança', 
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim([0.93, 0.97])

plt.tight_layout()
plt.show()

print(f"\nA proporção converge para {CONFIDENCE_LEVEL} conforme aumentamos o número de simulações.")
print(f"Isso demonstra empiricamente o significado do nível de confiança!")

## Conclusão

A simulação demonstra que:

1. **Aproximadamente 95% dos intervalos de confiança** calculados contêm a verdadeira média populacional (μ = 180)
2. **O nível de confiança não é sobre um IC específico**, mas sim sobre o método de construção do intervalo
3. **Se repetirmos o processo infinitas vezes**, esperamos que 95% dos intervalos contenham μ
4. **Cerca de 5% dos intervalos NÃO contêm μ**, e isso é esperado e normal!

Esta é a **interpretação frequentista** do intervalo de confiança: é uma afirmação sobre o método, não sobre um intervalo específico.